# Notebook 0 / R0 - Final Data Protocol Setup

Notebook ini mengunci protokol data final sebelum eksperimen R1-R10 dijalankan.

R0 bukan notebook training dan bukan tempat mencari setup terbaik. Output R0 adalah dataset inventory, class mapping, duplicate-conflict exclusion list, split manifest 70/15/15 untuk semua seed, dan ringkasan split. Pilot tuning baru dilakukan setelah R0, menggunakan validation set.

## 0. Rules

- Train set hanya untuk training.
- Validation set untuk checkpoint selection dan pilot hyperparameter tuning.
- Independent test set hanya untuk final reporting.
- Test set tidak boleh dipakai untuk memilih model, epoch, `temperature`, `alpha`, CORD loss weight, atau konfigurasi training lain.
- Semua model pada seed yang sama wajib memakai split manifest yang sama.
- Exact duplicate image dengan label berbeda dikeluarkan sebelum split untuk mencegah label ambiguity dan cross-split leakage.

In [1]:
# ============================================================
# 1. Imports
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import random
import textwrap

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

In [2]:
# ============================================================
# 2. Configuration
# ============================================================

KAGGLE_DATASET_DIR = Path("/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized")
LOCAL_DATASET_CANDIDATES = [
    Path.cwd() / "dataset-resized",
    Path.cwd() / "data" / "dataset-resized",
    Path.cwd() / "trashnet" / "dataset-resized",
]

DEFAULT_OUTPUT_DIR = (
    Path("/kaggle/working/final_research/r0_data_protocol")
    if Path("/kaggle/working").exists()
    else Path.cwd() / "final_research" / "r0_data_protocol"
)

DATASET_DIR = Path(os.environ.get("TRASHNET_DATASET_DIR", str(KAGGLE_DATASET_DIR)))
if not DATASET_DIR.exists():
    for candidate in LOCAL_DATASET_CANDIDATES:
        if candidate.exists():
            DATASET_DIR = candidate
            break

OUTPUT_DIR = Path(os.environ.get("R0_OUTPUT_DIR", str(DEFAULT_OUTPUT_DIR)))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
CLASS_TO_IDX = {class_name: idx for idx, class_name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: class_name for class_name, idx in CLASS_TO_IDX.items()}

EXPECTED_CLASS_COUNTS = {
    "cardboard": 403,
    "glass": 501,
    "metal": 410,
    "paper": 594,
    "plastic": 482,
    "trash": 137,
}

SEEDS = [42, 123, 777, 2026, 3407]
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
COMPUTE_FILE_HASHES = True
STRICT_EXPECTED_COUNTS = True
EXCLUDE_CONFLICTING_DUPLICATES = True

print(f"Dataset dir : {DATASET_DIR}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Seeds       : {SEEDS}")

Dataset dir : /kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized
Output dir  : /kaggle/working/final_research/r0_data_protocol
Seeds       : [42, 123, 777, 2026, 3407]


In [3]:
# ============================================================
# 3. Reproducibility Helpers
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def write_json(path: Path, data: dict) -> None:
    path.write_text(json.dumps(data, indent=2, sort_keys=False), encoding="utf-8")


seed_everything(SEEDS[0])

## 1. Resolve Dataset and Class Folders

Cell berikut memastikan folder dataset tersedia dan semua folder kelas final bisa ditemukan. Matching folder dibuat case-insensitive agar tetap aman jika nama folder memakai kapitalisasi berbeda.

In [4]:
# ============================================================
# 4. Resolve Class Directories
# ============================================================

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        "Dataset directory not found. Set TRASHNET_DATASET_DIR or update DATASET_DIR. "
        f"Current value: {DATASET_DIR}"
    )

available_dirs = {p.name.lower(): p for p in DATASET_DIR.iterdir() if p.is_dir()}
class_dirs = {}
missing_classes = []

for class_name in CLASS_NAMES:
    class_dir = available_dirs.get(class_name.lower())
    if class_dir is None:
        missing_classes.append(class_name)
    else:
        class_dirs[class_name] = class_dir

if missing_classes:
    raise ValueError(
        "Missing expected class folders: "
        + ", ".join(missing_classes)
        + f". Available folders: {sorted(available_dirs)}"
    )

class_dirs

{'cardboard': PosixPath('/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized/cardboard'),
 'glass': PosixPath('/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized/glass'),
 'metal': PosixPath('/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized/metal'),
 'paper': PosixPath('/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized/paper'),
 'plastic': PosixPath('/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized/plastic'),
 'trash': PosixPath('/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized/trash')}

## 2. Build Dataset Inventory

Inventory menyimpan metadata dasar setiap gambar. File rusak akan terdeteksi di sini. Untuk final experiment, file rusak tidak boleh dibiarkan lolos ke split manifest.

In [5]:
# ============================================================
# 5. Inventory Helpers
# ============================================================

def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def inspect_image(path: Path) -> dict:
    try:
        with Image.open(path) as image:
            width, height = image.size
            mode = image.mode
            image.verify()
        return {
            "is_valid": True,
            "width": width,
            "height": height,
            "mode": mode,
            "error": "",
        }
    except Exception as exc:
        return {
            "is_valid": False,
            "width": None,
            "height": None,
            "mode": None,
            "error": repr(exc),
        }


def normalized_relative_path(path: Path, root: Path) -> str:
    return path.relative_to(root).as_posix()

In [6]:
# ============================================================
# 6. Create Dataset Inventory
# ============================================================

records = []

for class_name in CLASS_NAMES:
    class_dir = class_dirs[class_name]
    image_paths = sorted(
        p for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    for image_path in tqdm(image_paths, desc=f"Inventory {class_name}"):
        image_info = inspect_image(image_path)
        sha256 = file_sha256(image_path) if COMPUTE_FILE_HASHES and image_info["is_valid"] else ""
        records.append({
            "image_path": str(image_path),
            "relative_path": normalized_relative_path(image_path, DATASET_DIR),
            "filename": image_path.name,
            "extension": image_path.suffix.lower(),
            "label": class_name,
            "class_id": CLASS_TO_IDX[class_name],
            "file_size_bytes": image_path.stat().st_size,
            "sha256": sha256,
            **image_info,
        })

inventory_df = pd.DataFrame(records)
inventory_df = inventory_df.sort_values(["class_id", "relative_path"]).reset_index(drop=True)
inventory_df.insert(0, "sample_id", [f"trashnet_{idx:05d}" for idx in range(len(inventory_df))])

inventory_path = OUTPUT_DIR / "dataset_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)

print(f"Saved: {inventory_path}")
display(inventory_df.head())
display(inventory_df.tail())

Inventory cardboard:   0%|          | 0/403 [00:00<?, ?it/s]

Inventory glass:   0%|          | 0/501 [00:00<?, ?it/s]

Inventory metal:   0%|          | 0/410 [00:00<?, ?it/s]

Inventory paper:   0%|          | 0/594 [00:00<?, ?it/s]

Inventory plastic:   0%|          | 0/482 [00:00<?, ?it/s]

Inventory trash:   0%|          | 0/137 [00:00<?, ?it/s]

Saved: /kaggle/working/final_research/r0_data_protocol/dataset_inventory.csv


,sample_id,image_path,relative_path,filename,extension,label,class_id,file_size_bytes,sha256,is_valid,width,height,mode,error
0,trashnet_00000,/kaggle/input/datasets/feyzazkefe/trashnet/dat...,cardboard/cardboard1.jpg,cardboard1.jpg,.jpg,cardboard,0,17333,9573fd98a5aca2ab9a4c5b9ebcdfa4dcf9719b216c2025...,True,512,384,RGB,
1,trashnet_00001,/kaggle/input/datasets/feyzazkefe/trashnet/dat...,cardboard/cardboard10.jpg,cardboard10.jpg,.jpg,cardboard,0,21683,f460e7a993aabafef6839fa1fce1034c40713411beff32...,True,512,384,RGB,
2,trashnet_00002,/kaggle/input/datasets/feyzazkefe/trashnet/dat...,cardboard/cardboard100.jpg,cardboard100.jpg,.jpg,cardboard,0,14884,13277fa7ae788f6f342a7d41ada36d4adcd20c58e504f2...,True,512,384,RGB,
3,trashnet_00003,/kaggle/input/datasets/feyzazkefe/trashnet/dat...,cardboard/cardboard101.jpg,cardboard101.jpg,.jpg,cardboard,0,14289,2f3dafed35830c61c3464c876f7cfb82ad3b635078f630...,True,512,384,RGB,
4,trashnet_00004,/kaggle/input/datasets/feyzazkefe/trashnet/dat...,cardboard/cardboard102.jpg,cardboard102.jpg,.jpg,cardboard,0,18015,5a88bce0103f59145f02b7fc0f50aa6dc3fbf29a1ffcae...,True,512,384,RGB,


,sample_id,image_path,relative_path,filename,extension,label,class_id,file_size_bytes,sha256,is_valid,width,height,mode,error
2522,trashnet_02522,/kaggle/input/datasets/feyzazkefe/trashnet/dat...,trash/trash95.jpg,trash95.jpg,.jpg,trash,5,7555,605ecba52073b4fdeb9f5dd0f9a635963b7fd8762f96c2...,True,512,384,RGB,
2523,trashnet_02523,/kaggle/input/datasets/feyzazkefe/trashnet/dat...,trash/trash96.jpg,trash96.jpg,.jpg,trash,5,16664,5bfe21d663958b01b145a4934d211f721beb68db99b0be...,True,512,384,RGB,
2524,trashnet_02524,/kaggle/input/datasets/feyzazkefe/trashnet/dat...,trash/trash97.jpg,trash97.jpg,.jpg,trash,5,9135,9bc6671a5dcee161730495154c2aa1d174366f456f0d05...,True,512,384,RGB,
2525,trashnet_02525,/kaggle/input/datasets/feyzazkefe/trashnet/dat...,trash/trash98.jpg,trash98.jpg,.jpg,trash,5,14558,1c04329bd1a226bf2498f4b4693d97eeaa6531da1ea2f9...,True,512,384,RGB,
2526,trashnet_02526,/kaggle/input/datasets/feyzazkefe/trashnet/dat...,trash/trash99.jpg,trash99.jpg,.jpg,trash,5,11607,782cf3677ff3368ec77f9e7dc47594d8431f28417486e6...,True,512,384,RGB,


In [7]:
# ============================================================
# 7. Inventory Summary and Checks
# ============================================================

class_count_df = (
    inventory_df.groupby(["label", "class_id"], as_index=False)
    .agg(num_images=("sample_id", "count"))
    .sort_values("class_id")
)

class_count_df["expected_images"] = class_count_df["label"].map(EXPECTED_CLASS_COUNTS)
class_count_df["matches_expected"] = class_count_df["num_images"] == class_count_df["expected_images"]

invalid_df = inventory_df[~inventory_df["is_valid"]].copy()
duplicate_hash_df = inventory_df[
    inventory_df["sha256"].ne("") & inventory_df.duplicated("sha256", keep=False)
].sort_values("sha256")
duplicate_conflict_hashes = (
    duplicate_hash_df.groupby("sha256")
    .filter(lambda group: group["label"].nunique() > 1)["sha256"]
    .unique()
)
duplicate_conflict_df = duplicate_hash_df[
    duplicate_hash_df["sha256"].isin(duplicate_conflict_hashes)
].copy()
duplicate_same_label_df = duplicate_hash_df[
    ~duplicate_hash_df["sha256"].isin(duplicate_conflict_hashes)
].copy()
excluded_sample_ids = set(duplicate_conflict_df["sample_id"])

class_count_path = OUTPUT_DIR / "dataset_class_counts.csv"
invalid_path = OUTPUT_DIR / "invalid_images.csv"
duplicates_path = OUTPUT_DIR / "duplicate_candidates.csv"
duplicate_conflicts_path = OUTPUT_DIR / "excluded_duplicate_conflicts.csv"
duplicate_same_label_path = OUTPUT_DIR / "duplicate_same_label_candidates.csv"

class_count_df.to_csv(class_count_path, index=False)
invalid_df.to_csv(invalid_path, index=False)
duplicate_hash_df.to_csv(duplicates_path, index=False)
duplicate_conflict_df.to_csv(duplicate_conflicts_path, index=False)
duplicate_same_label_df.to_csv(duplicate_same_label_path, index=False)

display(class_count_df)
print(f"Total images       : {len(inventory_df)}")
print(f"Invalid images     : {len(invalid_df)}")
print(f"Duplicate hashes   : {len(duplicate_hash_df)}")
print(f"Conflict duplicates: {len(duplicate_conflict_df)}")
print(f"Same-label duplicates: {len(duplicate_same_label_df)}")

if len(invalid_df) > 0:
    raise ValueError(f"Found {len(invalid_df)} invalid images. Check {invalid_path}")

if len(duplicate_conflict_df) > 0:
    print("\nExact duplicate images with conflicting labels will be excluded before splitting:")
    display(duplicate_conflict_df[["sample_id", "label", "relative_path", "sha256"]])

if STRICT_EXPECTED_COUNTS and not class_count_df["matches_expected"].all():
    raise ValueError(
        "Class counts do not match expected TrashNet counts. "
        f"Check {class_count_path} and verify the dataset source."
    )

,label,class_id,num_images,expected_images,matches_expected
0,cardboard,0,403,403,True
1,glass,1,501,501,True
2,metal,2,410,410,True
3,paper,3,594,594,True
4,plastic,4,482,482,True
5,trash,5,137,137,True


Total images       : 2527
Invalid images     : 0
Duplicate hashes   : 6
Conflict duplicates: 6
Same-label duplicates: 0

Exact duplicate images with conflicting labels will be excluded before splitting:


,sample_id,label,relative_path,sha256
421,trashnet_00421,glass,glass/glass115.jpg,81546d1362d75fc60718e5852593c58f66f813e8c1e9d1...
1305,trashnet_01305,metal,metal/metal91.jpg,81546d1362d75fc60718e5852593c58f66f813e8c1e9d1...
488,trashnet_00488,glass,glass/glass176.jpg,c41b99aec8a3257c8668a2144ebbaf8042018f0a85d88e...
1967,trashnet_01967,plastic,plastic/plastic152.jpg,c41b99aec8a3257c8668a2144ebbaf8042018f0a85d88e...
724,trashnet_00724,glass,glass/glass389.jpg,e971f4f8f50e960e454ea724cec922fc2988bbf4027aa4...
2167,trashnet_02167,plastic,plastic/plastic332.jpg,e971f4f8f50e960e454ea724cec922fc2988bbf4027aa4...


## 3. Save Class Mapping

Class mapping ini harus dipakai konsisten oleh semua notebook training dan evaluation.

In [8]:
# ============================================================
# 8. Save Class Mapping
# ============================================================

generated_at_utc = datetime.now(timezone.utc).isoformat()

class_mapping = {
    "dataset_name": "TrashNet",
    "dataset_dir": str(DATASET_DIR),
    "source": "https://www.kaggle.com/datasets/feyzazkefe/trashnet",
    "generated_at_utc": generated_at_utc,
    "num_classes": len(CLASS_NAMES),
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "idx_to_class": {str(k): v for k, v in IDX_TO_CLASS.items()},
}

class_mapping_path = OUTPUT_DIR / "class_mapping.json"
write_json(class_mapping_path, class_mapping)

print(f"Saved: {class_mapping_path}")
class_mapping

Saved: /kaggle/working/final_research/r0_data_protocol/class_mapping.json


{'dataset_name': 'TrashNet',
 'dataset_dir': '/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized',
 'source': 'https://www.kaggle.com/datasets/feyzazkefe/trashnet',
 'generated_at_utc': '2026-06-12T11:47:37.286006+00:00',
 'num_classes': 6,
 'class_names': ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash'],
 'class_to_idx': {'cardboard': 0,
  'glass': 1,
  'metal': 2,
  'paper': 3,
  'plastic': 4,
  'trash': 5},
 'idx_to_class': {'0': 'cardboard',
  '1': 'glass',
  '2': 'metal',
  '3': 'paper',
  '4': 'plastic',
  '5': 'trash'}}

## 4. Create Stratified 70/15/15 Splits

Setiap seed menghasilkan satu manifest. Semua model pada seed tersebut memakai manifest yang sama.

In [9]:
# ============================================================
# 9. Split Helpers
# ============================================================

def make_stratified_split(df: pd.DataFrame, seed: int) -> pd.DataFrame:
    if abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) > 1e-9:
        raise ValueError("TRAIN_RATIO + VAL_RATIO + TEST_RATIO must equal 1.0")

    train_df, temp_df = train_test_split(
        df,
        train_size=TRAIN_RATIO,
        random_state=seed,
        stratify=df["label"],
        shuffle=True,
    )

    relative_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=relative_test_ratio,
        random_state=seed + 10_000,
        stratify=temp_df["label"],
        shuffle=True,
    )

    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"

    split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    split_df["seed"] = seed
    split_df["split_rank"] = split_df["split"].map({"train": 0, "val": 1, "test": 2})
    split_df = split_df.sort_values(["split_rank", "class_id", "relative_path"]).drop(columns="split_rank")
    split_df = split_df.reset_index(drop=True)

    return split_df


def validate_split(split_df: pd.DataFrame, seed: int, expected_total: int) -> None:
    if len(split_df) != expected_total:
        raise AssertionError(f"Seed {seed}: split size mismatch")

    if split_df["sample_id"].nunique() != expected_total:
        raise AssertionError(f"Seed {seed}: sample appears more than once")

    split_sets = {
        split_name: set(split_df.loc[split_df["split"] == split_name, "sample_id"])
        for split_name in ["train", "val", "test"]
    }

    if split_sets["train"] & split_sets["val"]:
        raise AssertionError(f"Seed {seed}: train/val leakage")
    if split_sets["train"] & split_sets["test"]:
        raise AssertionError(f"Seed {seed}: train/test leakage")
    if split_sets["val"] & split_sets["test"]:
        raise AssertionError(f"Seed {seed}: val/test leakage")

    for split_name in ["train", "val", "test"]:
        classes_in_split = set(split_df.loc[split_df["split"] == split_name, "label"])
        missing = set(CLASS_NAMES) - classes_in_split
        if missing:
            raise AssertionError(f"Seed {seed}: {split_name} missing classes {sorted(missing)}")

In [10]:
# ============================================================
# 10. Generate and Save Split Manifests
# ============================================================

manifest_columns = [
    "sample_id",
    "image_path",
    "relative_path",
    "label",
    "class_id",
    "split",
    "seed",
    "sha256",
]

all_manifest_dfs = []
valid_inventory_df = inventory_df[inventory_df["is_valid"]].copy()

if EXCLUDE_CONFLICTING_DUPLICATES:
    valid_inventory_df = valid_inventory_df[
        ~valid_inventory_df["sample_id"].isin(excluded_sample_ids)
    ].copy()
elif len(duplicate_conflict_df) > 0:
    raise ValueError(
        "Conflicting exact duplicates exist. Set EXCLUDE_CONFLICTING_DUPLICATES=True "
        "or define a manual duplicate policy before splitting."
    )

final_class_count_df = (
    valid_inventory_df.groupby(["label", "class_id"], as_index=False)
    .agg(num_images=("sample_id", "count"))
    .sort_values("class_id")
)
final_class_count_path = OUTPUT_DIR / "final_dataset_class_counts.csv"
final_class_count_df.to_csv(final_class_count_path, index=False)

print(f"Raw valid images       : {len(inventory_df[inventory_df['is_valid']])}")
print(f"Excluded conflicts     : {len(excluded_sample_ids)}")
print(f"Final split candidates : {len(valid_inventory_df)}")
display(final_class_count_df)

for seed in SEEDS:
    seed_everything(seed)
    split_df = make_stratified_split(valid_inventory_df, seed)
    validate_split(split_df, seed=seed, expected_total=len(valid_inventory_df))

    manifest_df = split_df[manifest_columns].copy()
    manifest_path = OUTPUT_DIR / f"split_manifest_seed_{seed}.csv"
    manifest_df.to_csv(manifest_path, index=False)

    all_manifest_dfs.append(manifest_df)
    print(f"Saved: {manifest_path}")

all_manifests_df = pd.concat(all_manifest_dfs, ignore_index=True)
all_manifest_path = OUTPUT_DIR / "split_manifest_all_seeds.csv"
all_manifests_df.to_csv(all_manifest_path, index=False)

print(f"Saved: {all_manifest_path}")

Raw valid images       : 2527
Excluded conflicts     : 6
Final split candidates : 2521


,label,class_id,num_images
0,cardboard,0,403
1,glass,1,498
2,metal,2,409
3,paper,3,594
4,plastic,4,480
5,trash,5,137


Saved: /kaggle/working/final_research/r0_data_protocol/split_manifest_seed_42.csv
Saved: /kaggle/working/final_research/r0_data_protocol/split_manifest_seed_123.csv
Saved: /kaggle/working/final_research/r0_data_protocol/split_manifest_seed_777.csv
Saved: /kaggle/working/final_research/r0_data_protocol/split_manifest_seed_2026.csv
Saved: /kaggle/working/final_research/r0_data_protocol/split_manifest_seed_3407.csv
Saved: /kaggle/working/final_research/r0_data_protocol/split_manifest_all_seeds.csv


In [11]:
# ============================================================
# 11. Split Summary
# ============================================================

split_summary_df = (
    all_manifests_df.groupby(["seed", "split", "label", "class_id"], as_index=False)
    .agg(num_images=("sample_id", "count"))
    .sort_values(["seed", "split", "class_id"])
)

split_totals_df = (
    all_manifests_df.groupby(["seed", "split"], as_index=False)
    .agg(num_images=("sample_id", "count"))
)
split_totals_df["ratio"] = split_totals_df["num_images"] / len(valid_inventory_df)

split_summary_path = OUTPUT_DIR / "split_summary.csv"
split_totals_path = OUTPUT_DIR / "split_totals.csv"

split_summary_df.to_csv(split_summary_path, index=False)
split_totals_df.to_csv(split_totals_path, index=False)

print(f"Saved: {split_summary_path}")
print(f"Saved: {split_totals_path}")

display(split_totals_df)
display(split_summary_df.head(30))

Saved: /kaggle/working/final_research/r0_data_protocol/split_summary.csv
Saved: /kaggle/working/final_research/r0_data_protocol/split_totals.csv


,seed,split,num_images,ratio
0,42,test,379,0.150337
1,42,train,1764,0.699722
2,42,val,378,0.149940
3,123,test,379,0.150337
4,123,train,1764,0.699722
5,123,val,378,0.149940
6,777,test,379,0.150337
7,777,train,1764,0.699722
8,777,val,378,0.149940
9,2026,test,379,0.150337


,seed,split,label,class_id,num_images
0,42,test,cardboard,0,61
1,42,test,glass,1,75
2,42,test,metal,2,62
3,42,test,paper,3,89
4,42,test,plastic,4,72
5,42,test,trash,5,20
6,42,train,cardboard,0,282
7,42,train,glass,1,348
8,42,train,metal,2,286
9,42,train,paper,3,416


## 5. Final Integrity Gate

Cell ini sengaja dibuat tegas. Kalau gagal, jangan lanjut ke R1-R10 sebelum penyebabnya dibetulkan.

In [12]:
# ============================================================
# 12. Final Integrity Gate
# ============================================================

required_artifacts = [
    OUTPUT_DIR / "class_mapping.json",
    OUTPUT_DIR / "dataset_inventory.csv",
    OUTPUT_DIR / "dataset_class_counts.csv",
    OUTPUT_DIR / "final_dataset_class_counts.csv",
    OUTPUT_DIR / "invalid_images.csv",
    OUTPUT_DIR / "duplicate_candidates.csv",
    OUTPUT_DIR / "duplicate_same_label_candidates.csv",
    OUTPUT_DIR / "excluded_duplicate_conflicts.csv",
    OUTPUT_DIR / "split_summary.csv",
    OUTPUT_DIR / "split_totals.csv",
    OUTPUT_DIR / "split_manifest_all_seeds.csv",
] + [OUTPUT_DIR / f"split_manifest_seed_{seed}.csv" for seed in SEEDS]

missing_artifacts = [path for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError(f"Missing R0 artifacts: {missing_artifacts}")

if len(invalid_df) != 0:
    raise AssertionError("Invalid images exist. R0 is not clean.")

if excluded_sample_ids & set(all_manifests_df["sample_id"]):
    raise AssertionError("Excluded duplicate-conflict samples still exist in split manifests.")

for seed in SEEDS:
    seed_df = all_manifests_df[all_manifests_df["seed"] == seed]
    validate_split(seed_df, seed=seed, expected_total=len(valid_inventory_df))

print("R0 integrity gate passed.")
print("Dataset, class mapping, and split manifests are ready for R1-R10.")

R0 integrity gate passed.
Dataset, class mapping, and split manifests are ready for R1-R10.


## 6. Save R0 Protocol Report

Report ini bisa dilampirkan atau dijadikan catatan eksekusi. Isinya bukan hasil model, hanya bukti setup data final.

In [13]:
# ============================================================
# 13. Save Protocol Report and Artifact Manifest
# ============================================================

artifact_manifest = {
    "stage": "R0 Final Data Protocol Setup",
    "generated_at_utc": generated_at_utc,
    "dataset_dir": str(DATASET_DIR),
    "output_dir": str(OUTPUT_DIR),
    "seeds": SEEDS,
    "split_ratios": {
        "train": TRAIN_RATIO,
        "val": VAL_RATIO,
        "test": TEST_RATIO,
    },
    "raw_valid_images": int(len(inventory_df[inventory_df["is_valid"]])),
    "excluded_duplicate_conflict_images": int(len(excluded_sample_ids)),
    "num_images": int(len(valid_inventory_df)),
    "num_classes": len(CLASS_NAMES),
    "class_names": CLASS_NAMES,
    "artifacts": [str(path) for path in required_artifacts],
    "notes": [
        "R0 is not a training experiment.",
        "Validation split is used for pilot tuning and checkpoint selection.",
        "Independent test split is locked until final reporting.",
        "Exact duplicate images with conflicting labels are excluded before splitting.",
    ],
}

artifact_manifest_path = OUTPUT_DIR / "r0_artifact_manifest.json"
write_json(artifact_manifest_path, artifact_manifest)

report_text = f"""
# R0 Final Data Protocol Report

Generated at UTC: `{generated_at_utc}`

## Dataset

- Dataset: TrashNet
- Dataset directory: `{DATASET_DIR}`
- Raw valid images: `{len(inventory_df[inventory_df['is_valid']])}`
- Excluded duplicate-conflict images: `{len(excluded_sample_ids)}`
- Final valid images after exclusion: `{len(valid_inventory_df)}`
- Classes: `{', '.join(CLASS_NAMES)}`

## Split Protocol

- Train: `{TRAIN_RATIO:.2f}`
- Validation: `{VAL_RATIO:.2f}`
- Independent test: `{TEST_RATIO:.2f}`
- Seeds: `{SEEDS}`

## Rule

R0 only creates the dataset inventory, class mapping, duplicate-conflict exclusion list, and split manifests. It does not select the best model setup. Exact duplicate images with conflicting labels are excluded before splitting to prevent label ambiguity and cross-split leakage. Hyperparameter tuning is done later in pilot runs using the validation split only. The independent test split is locked until final reporting.
""".strip()

report_path = OUTPUT_DIR / "r0_protocol_report.md"
report_path.write_text(report_text + "\n", encoding="utf-8")

print(f"Saved: {artifact_manifest_path}")
print(f"Saved: {report_path}")
print("\nR0 artifacts:")
for path in required_artifacts + [artifact_manifest_path, report_path]:
    print(f"- {path}")

Saved: /kaggle/working/final_research/r0_data_protocol/r0_artifact_manifest.json
Saved: /kaggle/working/final_research/r0_data_protocol/r0_protocol_report.md

R0 artifacts:
- /kaggle/working/final_research/r0_data_protocol/class_mapping.json
- /kaggle/working/final_research/r0_data_protocol/dataset_inventory.csv
- /kaggle/working/final_research/r0_data_protocol/dataset_class_counts.csv
- /kaggle/working/final_research/r0_data_protocol/final_dataset_class_counts.csv
- /kaggle/working/final_research/r0_data_protocol/invalid_images.csv
- /kaggle/working/final_research/r0_data_protocol/duplicate_candidates.csv
- /kaggle/working/final_research/r0_data_protocol/duplicate_same_label_candidates.csv
- /kaggle/working/final_research/r0_data_protocol/excluded_duplicate_conflicts.csv
- /kaggle/working/final_research/r0_data_protocol/split_summary.csv
- /kaggle/working/final_research/r0_data_protocol/split_totals.csv
- /kaggle/working/final_research/r0_data_protocol/split_manifest_all_seeds.csv
- /